# Lote F222 -- corrida por unidades con puntos de control en Drive

REGLAS DE ESTE CUADERNO, antes que el codigo:

1. **Ninguna credencial del fondo entra aqui.** El repo `trading-lotes` se clona
   con la sesion de GitHub **del usuario** (o se sube a mano un zip); Drive se
   monta con la sesion de Google **del usuario**. Ni la clave del VPS, ni la clave
   de despliegue `~/.ssh/github_lotes_ro`, ni ninguna clave de broker.
2. **El sello se verifica ANTES del primer sorteo.** Si el sha256 de la spec no es
   el del `.seal`, o el HEAD clonado no es el commit pactado, esta corrida se para
   sin escribir nada.
3. Esta verificacion es una COMODIDAD, no una garantia: quien controla esta VM
   controla tambien esta comprobacion. La verificacion que vale es la del VPS
   (`scripts/aceptar_lote.py`).
4. Lo que se escribe en Drive son DATOS: `.npy`, `.json`, `.sha256`, `.txt`.
   Nunca un `.pkl`, nunca un script, nunca nada ejecutable.

In [ ]:
# --- 1. parametros de la corrida (los rellena el operador humano) -------------
RUN_ID      = "piloto_sintetico_20260917"
COMMIT_SHA  = ""                      # sha EXACTO del commit de trading-lotes
REPO_URL    = "https://github.com/<usuario>/trading-lotes"
CARPETA_DRIVE = "/content/drive/MyDrive/lotes"
assert COMMIT_SHA, "sin el sha del commit no hay identidad de corrida: nunca una rama, nunca un tag"

In [ ]:
# --- 2. traer el codigo sellado ----------------------------------------------
# Opcion A (repo privado): autenticarse con la cuenta de GitHub DEL USUARIO.
# Opcion B: subir a mano un zip del subarbol y descomprimirlo en /content/trading-lotes.
# En ningun caso se pega aqui un token del fondo.
import os, subprocess
DESTINO = "/content/trading-lotes"
if not os.path.isdir(DESTINO):
    subprocess.run(["git", "clone", "--filter=blob:none", "--no-checkout", REPO_URL, DESTINO], check=True)
subprocess.run(["git", "-C", DESTINO, "fetch", "--depth", "1", "origin", COMMIT_SHA], check=True)
subprocess.run(["git", "-C", DESTINO, "checkout", COMMIT_SHA], check=True)
head = subprocess.run(["git", "-C", DESTINO, "rev-parse", "HEAD"],
                      capture_output=True, text=True, check=True).stdout.strip()
assert head == COMMIT_SHA, "HEAD %s != commit pactado %s" % (head, COMMIT_SHA)
print("codigo en", head)

In [ ]:
# --- 3. montar Drive (sesion de Google DEL USUARIO) --------------------------
from google.colab import drive
drive.mount("/content/drive")
import os
DIR_CORRIDA = os.path.join(CARPETA_DRIVE, RUN_ID)
os.makedirs(DIR_CORRIDA, exist_ok=True)
print("puntos de control en", DIR_CORRIDA)

In [ ]:
# --- 4. VERIFICAR EL SELLO ANTES DEL PRIMER SORTEO ---------------------------
import hashlib, json, sys
sys.path.insert(0, DESTINO)
from motor import nucleo

ruta_spec = os.path.join(DESTINO, "spec", RUN_ID + ".json")
if not os.path.exists(ruta_spec):
    ruta_spec = os.path.join(DESTINO, "piloto", "spec_" + RUN_ID.split("_", 1)[0] + "_sintetico.json")
crudo = open(ruta_spec, "rb").read()
sha_spec = hashlib.sha256(crudo).hexdigest()
sello = nucleo.leer_sello(ruta_spec + ".seal")
assert sello["sha256"] == sha_spec, "SPEC ADULTERADA: %s != %s" % (sha_spec, sello["sha256"])
spec = json.loads(crudo.decode("utf-8"))
assert len(spec["code_commit"]) == 40 and set(spec["code_commit"]) <= set("0123456789abcdef"), "code_commit no es un sha de 40 hex: %r" % spec["code_commit"]   # el sello se sube DESPUES del codigo: sello["commit"]=%s y COMMIT_SHA pueden diferir; quien manda es code_subtree (celda siguiente)
# 4.1 ARRANQUE DEL RUNTIME, ANTES de validar: un motor se registra en
#     nucleo.CALCULOS AL IMPORTARSE (lotes/motor/tortugas.py). valida_spec
#     rechaza con "calculo no registrado" toda spec cuyo `calculo` no este YA
#     en CALCULOS, asi que arrancar despues (celda 5) hacia imposible verificar
#     el sello (F227 pase 6, observacion 7 del PASO 5 de progress/impl_f227.md).
import arranque            # lotes/arranque.py (en el espejo cuelga de la raiz)
print("arranque:", arranque.importar_arranque(spec, registrar=print))
nucleo.valida_spec(spec)
nucleo.verificar_subarbol(spec, DESTINO)   # el codigo que corre ES el sellado
print("sello OK", sha_spec, "| unidades:", spec["unidades"], "| calculo:", spec["calculo"])

In [ ]:
# --- 5. correr (reanudable: volver a ejecutar esta celda no repite trabajo) ---
# 5.0 ARRANQUE DEL RUNTIME: ya se hizo en la celda 4, ANTES de valida_spec
#     (que exige el calculo YA registrado en nucleo.CALCULOS). Esta repeticion
#     es IDEMPOTENTE -- importlib devuelve el modulo que ya esta en sys.modules
#     y no vuelve a registrar nada -- y se queda porque ESTA celda es la
#     reanudable: quien la reejecute sola no puede sortear sin motor.
import arranque            # lotes/arranque.py (en el espejo cuelga de la raiz)
print("arranque:", arranque.importar_arranque(spec, registrar=print))

import time
registro = open(os.path.join(DIR_CORRIDA, "LOG.txt"), "a")
def apuntar(linea):
    registro.write(linea + "
"); registro.flush(); print(linea)

t0 = time.time()
resumen = nucleo.correr_lote(spec, DIR_CORRIDA, registrar=apuntar)
registro.close()
print("calculadas:", resumen["calculadas"])
print("reusadas  :", resumen["reusadas"])
print("recomputadas por corrupcion:", resumen["recomputadas_por_corrupcion"])
print("segundos de esta sesion: %.1f" % (time.time() - t0))

In [ ]:
# --- 6. manifiesto + su sha (esta es la cifra que se lleva al VPS) -----------
ruta_man = nucleo.escribir_manifiesto(spec, DIR_CORRIDA, sha_spec, sello, resumen)
sha_man = open(os.path.join(DIR_CORRIDA, "manifest.json.sha256")).read().split()[0]
print("manifest.json sha256 =", sha_man)
print("Pega ESE sha en el mensaje al VPS. Si no cuadra al llegar, el lote se rechaza entero.")

## Y ahora, en el VPS

    ./venv/bin/python scripts/bajar_lote_drive.py --folder-id <ID> --destino <dir>
    ./venv/bin/python scripts/aceptar_lote.py --run-dir <dir>         --spec lotes/spec/<run_id>.json --dry-run
    # y, si el seco pasa, la misma orden sin --dry-run

El lote no vale nada hasta que esa orden lo acepta: el manifiesto solo prueba el
TRANSPORTE. Lo que prueba el calculo es el recalculo de unidades sorteadas en el
VPS (docs/DISENO_COLAB_LOTES.md seccion 4.3).